In [1]:
from google.colab import drive

# Check if the drive is already mounted
try:
  drive.mount('/content/drive')
except ValueError:
  print("Drive is already mounted. Skipping mounting.")

Mounted at /content/drive


In [2]:
from google.colab import files
uploaded = files.upload() # Removed the extra indentation before this line

Saving plant_vs_other_model.h5 to plant_vs_other_model.h5


In [3]:
from google.colab import files
uploaded = files.upload() # Removed the extra indentation before this line

Saving final_project_plant_disease_model.keras to final_project_plant_disease_model.keras


In [4]:
!pip install flask ngrok tensorflow pyngrok # Added pyngrok to the installation command

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.7 MB/s eta 0:00:00


In [5]:
!ngrok authtoken 2wQScRXlu0iMhTikDLy8jruuyl3_3h2qquRPrG2exUrrXmngc # Replace <YOUR_AUTHTOKEN> with your actual authtoken

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# تثبيت الحزم (مرة واحدة فقط)
!pip install flask pyngrok tensorflow pillow

# استيراد المكتبات
from flask import Flask, request, render_template_string
from PIL import Image
import numpy as np
import tensorflow as tf
from pyngrok import ngrok
import io
import base64

# تحميل النماذج
plant_detector = tf.keras.models.load_model("/content/plant_vs_other_model.h5")
plant_classifier = tf.keras.models.load_model("/content/final_project_plant_disease_model.keras")

# أسماء الكلاسات
class_names = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry_(including_sour)_Powdery_mildew", "Cherry_(including_sour)_healthy",
    "Corn_(maize)_Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)Common_rust",
    "Corn_(maize)_Northern_Leaf_Blight", "Corn_(maize)_healthy", "Grape___Black_rot",
    "Grape__Esca(Black_Measles)", "Grape__Leaf_blight(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange__Haunglongbing(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
    "Pepper,bell__Bacterial_spot", "Pepper,bell__healthy", "Potato___Early_blight", "Potato___Late_blight",
    "Potato___healthy", "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot", "Tomato___Early_blight",
    "Tomato___Late_blight", "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
]

# تعليمات العناية
# --- تعليمات العناية أو العلاج ---
care_instructions = {
    "Apple___Apple_scab": "Use resistant varieties and apply fungicide early. Regularly inspect for signs of infection.",
    "Apple___Black_rot": "Remove infected branches and apply fungicide. Prune the tree to allow better air circulation.",
    "Apple___Cedar_apple_rust": "Remove nearby junipers and apply fungicides. Practice crop rotation.",
    "Apple___healthy": "No disease detected. Continue good orchard practices and regular inspections.",
    "Blueberry___healthy": "No disease detected. Maintain soil acidity and mulch regularly to retain moisture.",
    "Cherry_(including_sour)_Powdery_mildew": "Use sulfur-based fungicides and prune for better airflow. Avoid overhead watering.",
    "Cherry_(including_sour)_healthy": "No disease detected. Regularly prune and monitor for any new symptoms.",
    "Corn_(maize)_Cercospora_leaf_spot Gray_leaf_spot": "Use resistant varieties and fungicides. Avoid over-watering.",
    "Corn_(maize)Common_rust": "Plant resistant hybrids and apply fungicides if needed. Rotate crops to reduce disease spread.",
    "Corn_(maize)_Northern_Leaf_Blight": "Rotate crops and use resistant varieties. Apply fungicides as necessary.",
    "Corn_(maize)_healthy": "No disease detected. Continue practicing crop rotation and proper field management.",
    "Grape___Black_rot": "Remove mummified fruits and apply fungicide. Ensure proper spacing between vines for airflow.",
    "Grape__Esca(Black_Measles)": "Manage vine stress and prune properly. Keep vines healthy with good nutrition.",
    "Grape__Leaf_blight(Isariopsis_Leaf_Spot)": "Prune to increase airflow and apply fungicides. Regularly inspect the leaves for signs of infection.",
    "Grape___healthy": "No disease detected. Regular pruning and monitoring are essential for long-term health.",
    "Orange__Haunglongbing(Citrus_greening)": "Remove infected trees and control psyllid populations. Treat with systemic insecticides.",
    "Peach___Bacterial_spot": "Use disease-free seeds and copper-based sprays. Prune trees to improve air circulation.",
    "Peach___healthy": "No disease detected. Ensure proper pruning and adequate fertilization to maintain tree health.",
    "Pepper,bell__Bacterial_spot": "Use disease-free seeds and apply copper sprays. Avoid wetting leaves during watering.",
    "Pepper,bell__healthy": "No disease detected. Keep soil well-drained and use mulch to retain moisture.",
    "Potato___Early_blight": "Apply fungicide and remove affected foliage. Rotate crops to prevent disease recurrence.",
    "Potato___Late_blight": "Remove infected plants and use resistant varieties. Ensure proper spacing between plants.",
    "Potato___healthy": "No disease detected. Keep monitoring your crop and avoid overcrowding.",
    "Raspberry___healthy": "No disease detected. Prune canes and avoid wet foliage to reduce the risk of disease.",
    "Soybean___healthy": "No disease detected. Rotate crops and monitor regularly for pests and disease.",
    "Squash___Powdery_mildew": "Use sulfur fungicides and space plants for better airflow. Remove infected leaves promptly.",
    "Strawberry___Leaf_scorch": "Remove infected leaves and avoid overhead watering. Ensure good drainage around the plants.",
    "Strawberry___healthy": "No disease detected. Practice crop rotation and regular sanitation of planting areas.",
    "Tomato___Bacterial_spot": "Use copper-based sprays and avoid working with wet plants. Practice crop rotation.",
    "Tomato___Early_blight": "Remove infected leaves and apply fungicide. Ensure good spacing between plants for better airflow.",
    "Tomato___Late_blight": "Remove infected plants and use resistant varieties. Apply fungicides as needed.",
    "Tomato___Leaf_Mold": "Increase air circulation and apply fungicide. Avoid overhead watering.",
    "Tomato___Septoria_leaf_spot": "Avoid overhead watering and remove infected leaves. Use fungicides to control spread.",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Use insecticidal soap and maintain humidity around the plants.",
    "Tomato___Target_Spot": "Use proper fungicide and ensure good spacing between plants for airflow.",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Control whiteflies and remove infected plants. Disinfect tools regularly.",
    "Tomato___Tomato_mosaic_virus": "Remove infected plants and disinfect tools. Use virus-free seeds and resistant varieties.",
    "Tomato___healthy": "No disease detected. Continue good care practices and monitor for any potential issues."
}


# إعداد Flask
app = Flask(__name__)

# دوال تجهيز الصور
def preprocess_for_detector(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((150, 150))
    return np.expand_dims(np.array(img) / 255.0, axis=0)

def preprocess_for_classifier(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((224, 224))
    return np.expand_dims(np.array(img) / 255.0, axis=0)

# HTML محسّن بالكامل

...
# نفس كود النموذج والتهيئة السابق تمامًا
...

# HTML بعد التعديلات لتقسيم الصفحة لثلاثة أقسام
HTML_PAGE = '''
<!doctype html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>🌿 Plant Disease Detector</title>
    <style>
        body {
            font-family: 'Segoe UI', sans-serif;
            background: linear-gradient(to right, #f0fdf4, #e0f7fa);
            padding: 40px;
        }
        h1 {
            text-align: center;
            color: #14532d;
            font-size: 3em;
            margin-bottom: 40px;
        }
        .grid-container {
            display: grid;
            grid-template-columns: 1fr 1fr 1fr;
            gap: 40px;
        }
        .section-box {
            background: #ffffff;
            padding: 25px;
            border-radius: 20px;
            box-shadow: 0 8px 16px rgba(0,0,0,0.1);
            min-height: 400px;
        }
        form input[type="file"] {
            margin: 15px 0;
            padding: 12px;
            font-size: 1em;
            width: 100%;
        }
        form input[type="submit"] {
            background-color: #16a34a;
            color: white;
            border: none;
            padding: 14px 24px;
            font-size: 1.2em;
            border-radius: 10px;
            cursor: pointer;
            width: 100%;
        }
        .uploaded-image {
            max-width: 100%;
            border-radius: 10px;
            box-shadow: 0 0 10px #666;
        }
        .result, .care, .confidence-bar {
            margin-top: 15px;
            font-size: 18px;
        }
        .result.success { background: #4caf50; color: white; padding: 10px; border-radius: 10px; }
        .result.error { background: #e53935; color: white; padding: 10px; border-radius: 10px; }
        .progress-bar {
            width: 100%;
            background-color: #ddd;
            border-radius: 10px;
            height: 25px;
        }
        .progress-bar-fill {
            height: 100%;
            background-color: #4caf50;
            border-radius: 10px;
            text-align: center;
            color: white;
            font-weight: bold;
        }
        .loader {
    border: 6px solid #f3f3f3; /* Light grey */
    border-top: 6px solid #16a34a; /* Green */
    border-radius: 50%;
    width: 40px;
    height: 40px;
    animation: spin 1s linear infinite;
    margin: 20px auto; /* Center horizontally */
}

      footer {
            margin-top: auto;
            text-align: center;
            padding: 15px 0;
            font-size: 20px;
            color: #333;

        }
@keyframes spin {
    0% { transform: rotate(0deg); }
    100% { transform: rotate(360deg); }
}

    </style>
</head>
<body>
    <h1>🌿 Plant Disease Detection🌿</h1>
    <div class="grid-container">
        <!-- اختيار الصورة -->
        <div class="section-box">
            <form method="post" enctype="multipart/form-data" action="/predict" onsubmit="showLoader()">
                <label>📁Choose a plant image:</label><br>
                <input type="file" name="image" required><br>
                <input type="submit" value="🔍 Predict">
                <div id="loader" style="display:none;margin-top:10px;" class="loader"></div>
            </form>
        </div>

        <!-- عرض الصورة -->
        <div class="section-box">
            {% if uploaded_image %}
                <h3>🖼 Featured image:</h3>
                <img src="data:image/png;base64,{{ uploaded_image }}" class="uploaded-image">
            {% else %}
                <p>No image has been uploaded yet.</p>
            {% endif %}
        </div>

        <!-- نتائج التنبؤ -->
        <div class="section-box">
            {% if result %}
                <div class="result {{ result_class }}">{{ result }}</div>
            {% endif %}

            {% if file_name %}
                <div><strong>📄 Name:</strong> {{ file_name }}</div>
            {% endif %}

            {% if is_plant is not none %}
                <div><strong>🌱Is it a plant?</strong> {{ 'Yes✅' if is_plant else 'NO ❌' }}</div>
            {% endif %}

            {% if predicted_class %}
                <div><strong>🧬Predicted Class </strong> {{ predicted_class }}</div>
            {% endif %}

            {% if confidence %}
                <div><strong>📊Confidence:</strong> {{ confidence }}%</div>
                <div class="progress-bar">
                    <div class="progress-bar-fill" style="width: {{ confidence }}%">{{ confidence }}%</div>
                </div>
            {% endif %}

            {% if care %}
                <div class="care"><strong>📖Care Instructions</strong><br>{{ care }}</div>
            {% endif %}
        </div>
    </div>

    <script>
        function showLoader() {
            document.getElementById("loader").style.display = "block";
        }
    </script>
     <footer>
       🌸🌸 Made by samaa_abdelmohsen 🌸🌸
    </footer>
</body>
</html>
'''

...
# باقي الكود كما هو تمامًا مع صفحة /predict تعيد القيم بنفس الطريقة السابقة
...



# الصفحة الرئيسية
@app.route("/", methods=["GET"])
def home():
    return render_template_string(HTML_PAGE)

# صفحة التنبؤ
@app.route("/predict", methods=["POST"])
def predict():
    if 'image' not in request.files:
        return render_template_string(HTML_PAGE, result="❌ No image uploaded", result_class="error")

    file = request.files['image']
    try:
        img_bytes = file.read()
        encoded_img = base64.b64encode(img_bytes).decode('utf-8')
        file_name = file.filename

        detector_input = preprocess_for_detector(img_bytes)
        is_plant = plant_detector.predict(detector_input)[0][0] >= 0.5

        if not is_plant:
            return render_template_string(HTML_PAGE, result="❌ This is not a plant!", result_class="error", uploaded_image=encoded_img, file_name=file_name)

        classifier_input = preprocess_for_classifier(img_bytes)
        prediction = plant_classifier.predict(classifier_input)
        class_idx = int(np.argmax(prediction))
        class_name = class_names[class_idx]
        confidence = np.max(prediction) * 100
        care = care_instructions.get(class_name, "❗ There are no specific instructions for this condition.")

        return render_template_string(HTML_PAGE,
                                      result_class="success",
                                      result="✅ Plant detected and classified!",
                                      care=care,
                                      uploaded_image=encoded_img,
                                      file_name=file_name,
                                      is_plant=True,
                                      predicted_class=class_name,
                                      confidence=round(confidence, 2))

    except Exception as e:
        return render_template_string(HTML_PAGE, result=f"❌ Error: {str(e)}", result_class="error")

# ngrok
public_url = ngrok.connect(5000)
print(f" * Ngrok URL: {public_url}")

# تشغيل التطبيق
if __name__ == "__main__":
    app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)


 * Ngrok URL: NgrokTunnel: "https://23e1-34-168-141-77.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:03:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:03:39] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:04:08] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


...

INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:05:27] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:07:56] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:08:50] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:09:03] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:09:03] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:09:03] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:09:14] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:09:14] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:10:24] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:10:29] "GET /predict HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:10:30] "GET /predict HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [01/May/2025 12:10:30] "GET /predict HTTP/1.1" 405 -
